# loading

In [38]:
'''
conda deactivate
conda activate crane_test
srun --nodelist=comput76 --pty -c 4 --mem=50G jupyter lab --no-browser --port=8702 --ip=0.0.0.0
ssh 10.168.203.76 -L 8702:127.0.0.1:8702 #-R 37511:localhost:37511
'''

'\nconda deactivate\nconda activate crane_test\nsrun --nodelist=comput76 --pty -c 4 --mem=50G jupyter lab --no-browser --port=8702 --ip=0.0.0.0\nssh 10.168.203.76 -L 8702:127.0.0.1:8702 #-R 37511:localhost:37511\n'

In [39]:
'''
conda deactivate
conda activate crane_test_r
srun --nodelist=comput107 --pty -c 20 --mem=50G jupyter lab --no-browser --port=8703 --ip=0.0.0.0
ssh 10.168.203.107 -L 8703:127.0.0.1:8703 #-R 37511:localhost:37511
'''

'\nconda deactivate\nconda activate crane_test_r\nsrun --nodelist=comput107 --pty -c 20 --mem=50G jupyter lab --no-browser --port=8703 --ip=0.0.0.0\nssh 10.168.203.107 -L 8703:127.0.0.1:8703 #-R 37511:localhost:37511\n'

In [40]:
import os
import anndata as ad
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import re
%matplotlib inline

In [41]:
from IPython.core.interactiveshell import InteractiveShell
# InteractiveShell.ast_node_interactivity = 'all'  # 默认为'last'，即输出最后一个结果
InteractiveShell.ast_node_interactivity = 'all'

In [42]:
from cmapPy.pandasGEXpress.parse import parse  

In [43]:
import re

# 1. meta_data

## 1.1 细胞系信息

In [7]:
# 细胞系信息
cellline_meta = pd.read_csv('/public/home/caojun/project/crane_novel/2_true_data/input/cmap/cellline_info_beta.csv')

In [8]:
cellline_meta

,cell_iname,cellosaurus_id,donor_age,donor_age_death,donor_disease_age_onset,doubling_time,growth_medium,provider_catalog_id,feature_id,cell_type,donor_ethnicity,donor_sex,donor_tumor_phase,cell_lineage,primary_disease,subtype,provider_name,growth_pattern,ccle_name,cell_alias
0,1HAE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,normal,Unknown,Unknown,Unknown,unknown,unknown,normal fibroblast sample,NaN,unknown,NaN,NaN
1,AALE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,normal,Unknown,Unknown,Unknown,unknown,unknown,normal epithelium sample,NaN,unknown,NaN,NaN
2,AG06263_2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,tumor,Unknown,Unknown,Unknown,unknown,unknown,unknown,NaN,unknown,NaN,NaN
3,AG06840_A,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,tumor,Unknown,Unknown,Unknown,unknown,unknown,unknown,NaN,unknown,NaN,NaN
4,AG078N1_1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,tumor,Unknown,Unknown,Unknown,unknown,unknown,unknown,NaN,unknown,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235,RCC10RGB,CVCL_1647,NaN,NaN,NaN,108,DMEM+ 1%FBS,NaN,c-526,tumor,Unknown,M,Unknown,kidney,kidney cancer,carcinoma,RIKEN,adherent,RCC10RGB_KIDNEY,10RGB
236,OVK18,CVCL_3770,NaN,NaN,NaN,36,MEM + 1% FBS,NaN,c-148,tumor,Unknown,F,Unknown,ovary,ovarian cancer,carcinoma,RIKEN,adherent,OVK18_OVARY,OVK-18
237,JHUEM2,CVCL_4656,NaN,NaN,NaN,48,DMEM/F12 (Hyclone Cat.# SH323.1),NaN,c-187,tumor,Unknown,F,Primary,endometrium,endometrial cancer,carcinoma,RIKEN,adherent,JHUEM2_ENDOMETRIUM,JHUEM-2
238,OVCAR8,CVCL_1629,NaN,NaN,NaN,60,RPMI-164 ATCC catalog # 3-21,NaN,c-349,tumor,Unknown,F,Primary,ovary,ovarian cancer,carcinoma,NCI/DCTD,adherent,OVCAR8_OVARY,OVCAR-8|NIH:OVCAR-8


In [9]:
cellline_meta_sub = cellline_meta.loc[:,['cell_iname','cell_lineage','primary_disease','subtype','cell_alias']]
cellline_meta_sub.index = cellline_meta_sub['cell_iname'].copy()

In [10]:
#cellline_meta_sub.to_csv('/public/home/caojun/project/RUSH/3_work/input/cellline_meta_sub.csv')

In [11]:
cellline_meta_sub 

,cell_iname,cell_lineage,primary_disease,subtype,cell_alias
cell_iname,,,,,
1HAE,1HAE,unknown,unknown,normal fibroblast sample,NaN
AALE,AALE,unknown,unknown,normal epithelium sample,NaN
AG06263_2,AG06263_2,unknown,unknown,unknown,NaN
AG06840_A,AG06840_A,unknown,unknown,unknown,NaN
AG078N1_1,AG078N1_1,unknown,unknown,unknown,NaN
...,...,...,...,...,...
RCC10RGB,RCC10RGB,kidney,kidney cancer,carcinoma,10RGB
OVK18,OVK18,ovary,ovarian cancer,carcinoma,OVK-18
JHUEM2,JHUEM2,endometrium,endometrial cancer,carcinoma,JHUEM-2


In [ ]:
cellline_meta_sub['cell_lineage'].value_counts()

cell_lineage
unknown                               82
lung                                  32
haematopoietic_and_lymphoid_tissue    21
large_intestine                       19
ovary                                 15
breast                                11
endometrium                            9
skin                                   9
central_nervous_system                 7
prostate                               6
urinary_tract                          5
kidney                                 4
bone                                   4
liver                                  4
soft_tissue                            4
stomach                                3
placenta                               1
autonomic_ganglia                      1
cervix                                 1
pancreas                               1
upper_aerodigestive_tract              1
Name: count, dtype: int64

In [13]:
cellline_meta_sub.loc[cellline_meta_sub['cell_lineage'] == 'stomach',:]

,cell_iname,cell_lineage,primary_disease,subtype,cell_alias
cell_iname,,,,,
AGS,AGS,stomach,gastric cancer,adenocarcinoma,NaN
IM95,IM95,stomach,gastric cancer,adenocarcinoma,IM-95
MKN45,MKN45,stomach,gastric cancer,adenocarcinoma,MKN-45|MKN 45


In [14]:
cellline_meta_sub.loc[cellline_meta_sub['cell_lineage'] == 'lung',:]

,cell_iname,cell_lineage,primary_disease,subtype,cell_alias
cell_iname,,,,,
H1299,H1299,lung,lung cancer,non small cell carcinoma,NaN
H1975,H1975,lung,lung cancer,carcinoma,NaN
HCC515,HCC515,lung,lung cancer,carcinoma,HCC0515
SALE,SALE,lung,lung cancer,lung cancer,NaN
NL20,NL20,lung,normal lung sample,normal lung sample,NL-20
A549,A549,lung,lung cancer,non small cell carcinoma,A 549
HCC827,HCC827,lung,lung cancer,non small cell lung carcinoma,HCC-827
NCIH1437,NCIH1437,lung,lung cancer,non small cell lung carcinoma,H-1437|NCI-H1437
NCIH1563,NCIH1563,lung,lung cancer,non small cell lung carcinoma,H1563|H-1563|NCI-H1563


In [15]:
cellline_meta_sub.loc[cellline_meta_sub['cell_lineage'] == 'unknown','subtype'].value_counts()

subtype
unknown                                          69
normal stem cell sample                           4
normal epithelium sample                          3
normal fibroblast sample                          1
chronic myeloid leukemia (cml)                    1
hepatoma                                          1
normal umbilical vein sample                      1
normal prostate sample                            1
medullary cystic?kidney?disease (MCKD) type 1     1
Name: count, dtype: int64

In [16]:
list(cellline_meta_sub['cell_iname'])

['1HAE',
 'AALE',
 'AG06263_2',
 'AG06840_A',
 'AG078N1_1',
 'C42',
 'CD34',
 'HAP1',
 'HEK293T',
 'HIMG001',
 'HIMG002',
 'HME',
 'HMELZ',
 'HPTEC',
 'HS27A',
 'HUES3',
 'HUH751',
 'HUVEC',
 'IPS-15-2',
 'IPS-3',
 'IPS-ND34732',
 'LHSAR',
 'MCH58',
 'MCLF0067SZ',
 'MCLF015CN',
 'MCLF022CN',
 'MCLF027CN',
 'MCLF033SZ',
 'MCLF035SZ',
 'MCLF037SZ',
 'MCLF040SZ',
 'MCLF051CN',
 'MCLF054CN',
 'MCLF056CN',
 'MCLF076SZ',
 'MCLF100SZ',
 'MCLF117SZ',
 'MCLF123SZ',
 'MCLF126CN',
 'MCLF130CN',
 'MCLF135CN',
 'MCLF137CN',
 'MCLF141SZ',
 'MCLF162SZ',
 'MICROGLIA-PSEN1',
 'MNEU',
 'NAMEC8',
 'ND34732_7',
 'NKDBA',
 'P1A82',
 'P2X2B2',
 'SKB',
 'WA09',
 'XC.500',
 'XC.L10',
 'XC.L100',
 'XC.P026',
 'XC.P031',
 'XC.P033',
 'XC.P091',
 'XC.P092',
 'XC.P901',
 'XC.P904',
 'XC.P905',
 'XC.P906',
 'XC.P907',
 'XC.P908',
 'XC.P909',
 'XC.P910',
 'XC.P911',
 'XC.P912',
 'XC.P914',
 'XC.P915',
 'XC.P922',
 'XC.P930',
 'XC.P931',
 'XC.P932',
 'XC.P933',
 'XC.P934',
 'XC.P935',
 'XC.P936',
 'XC.R10',
 'HME1',

In [17]:
cellline_meta_sub.loc[cellline_meta_sub['cell_iname']=='A549',:]

,cell_iname,cell_lineage,primary_disease,subtype,cell_alias
cell_iname,,,,,
A549,A549,lung,lung cancer,non small cell carcinoma,A 549


## 1.2 基因信息

In [18]:
# 测量的基因型信息
gene_meta = pd.read_csv('/public/home/caojun/project/crane_novel/2_true_data/input/cmap/gene_info_beta.csv')
gene_meta.index = gene_meta['gene_id'].astype(str).copy() 

In [19]:
gene_meta

,gene_id,gene_symbol,ensembl_id,gene_title,gene_type,src,feature_space
gene_id,,,,,,,
750,750,GAS8-AS1,ENSG00000221819,GAS8 antisense RNA 1,ncRNA,NCBI,inferred
6315,6315,ATXN8OS,NaN,ATXN8 opposite strand lncRNA,ncRNA,NCBI,inferred
7503,7503,XIST,ENSG00000229807,X inactive specific transcript,ncRNA,NCBI,inferred
8552,8552,INE1,ENSG00000224975,inactivation escape 1,ncRNA,NCBI,inferred
9834,9834,FAM30A,ENSG00000226777,family with sequence similarity 30 member A,ncRNA,NCBI,inferred
...,...,...,...,...,...,...,...
100287932,100287932,TIMM23,ENSG00000265354,translocase of inner mitochondrial membrane 23,protein-coding,NCBI,best inferred
100289678,100289678,ZNF783,ENSG00000204946,zinc finger family member 783,protein-coding,NCBI,best inferred
100507436,100507436,MICA,ENSG00000204520,MHC class I polypeptide-related sequence A,protein-coding,NCBI,best inferred


## 1.3 数据集的设置组合信息

In [20]:
# 数据集的设置组合信息
tar_meta = pd.read_csv('/public/home/caojun/project/crane_novel/2_true_data/input/cmap/siginfo_beta.txt', sep='\t')

/tmp/ipykernel_30967/1937195739.py:2: DtypeWarning: Columns (0,3,4,28,29) have mixed types. Specify dtype option on import or set low_memory=False.
  tar_meta = pd.read_csv('/public/home/caojun/project/crane_novel/2_true_data/input/cmap/siginfo_beta.txt', sep='\t')


In [21]:
tar_meta

,bead_batch,nearest_dose,pert_dose,pert_dose_unit,pert_idose,pert_itime,pert_time,pert_time_unit,cell_mfc_name,pert_mfc_id,...,cell_iname,det_wells,det_plates,distil_ids,build_name,project_code,cmap_name,is_exemplar_sig,is_ncs_sig,is_null_sig
0,b17,NaN,100.0,ug/ml,100 ug/ml,336 h,336.0,h,N8,BRD-U44432129,...,NAMEC8,H05|H06|H07|H08,MET001_N8_XH_X1_B17,MET001_N8_XH_X1_B17:H05|MET001_N8_XH_X1_B17:H0...,NaN,MET,BRD-U44432129,0,0.0,0.0
1,b15,10.0,10.0,uM,10 uM,3 h,3.0,h,A549,BRD-K81418486,...,A549,L04|L08|L12,ABY001_A549_XH_X1_B15,ABY001_A549_XH_X1_B15:L04|ABY001_A549_XH_X1_B1...,NaN,ABY,vorinostat,0,1.0,0.0
2,b15,2.5,2.5,uM,2.5 uM,24 h,24.0,h,HT29,BRD-K70511574,...,HT29,E18|E22,ABY001_HT29_XH_X1_B15,ABY001_HT29_XH_X1_B15:E18|ABY001_HT29_XH_X1_B1...,NaN,ABY,HMN-214,0,1.0,0.0
3,b18,10.0,10.0,uM,10 uM,3 h,3.0,h,HME1,BRD-K81418486,...,HME1,F19,LTC002_HME1_3H_X1_B18,LTC002_HME1_3H_X1_B18:F19,NaN,LTC,vorinostat,0,0.0,0.0
4,b15,10.0,10.0,uM,10 uM,3 h,3.0,h,H1975,BRD-A61304759,...,H1975,P01|P05|P09,ABY001_H1975_XH_X1_B15,ABY001_H1975_XH_X1_B15:P01|ABY001_H1975_XH_X1_...,NaN,ABY,tanespimycin,0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1201939,b18,10.0,10.0,uM,10 uM,24 h,24.0,h,HCC515,BRD-K48853221,...,HCC515,K01,DOSVAL001_HCC515_24H_X1_B18|DOSVAL001_HCC515_2...,DOSVAL001_HCC515_24H_X1_B18:K01|DOSVAL001_HCC5...,NaN,DOSVAL,BRD-K48853221,1,1.0,0.0
1201940,b18,10.0,10.0,uM,10 uM,24 h,24.0,h,HCC515,BRD-K90382497,...,HCC515,O03,DOSVAL001_HCC515_24H_X1_B18|DOSVAL001_HCC515_2...,DOSVAL001_HCC515_24H_X1_B18:O03|DOSVAL001_HCC5...,NaN,DOSVAL,GW-843682X,0,1.0,0.0
1201941,b19,20.0,20.0,uM,20 uM,24 h,24.0,h,HCC515,BRD-K45785972,...,HCC515,M22,DOSVAL002_HCC515_24H_X1.L2_B19|DOSVAL002_HCC51...,DOSVAL002_HCC515_24H_X1.L2_B19:M22|DOSVAL002_H...,NaN,DOSVAL,BRD-K45785972,0,1.0,0.0
1201942,b19,4.0,5.0,uM,4 uM,24 h,24.0,h,A375,BRD-K28513938,...,A375,E09,DOSVAL004_A375_24H_X1.A2_B19|DOSVAL004_A375_24...,DOSVAL004_A375_24H_X1.A2_B19:E09|DOSVAL004_A37...,NaN,DOSVAL,BRD-K28513938,0,1.0,0.0


In [22]:
tar_meta['pert_type'].value_counts()

pert_type
trt_cp             720216
trt_sh             177263
trt_xpr            140945
ctl_vehicle         39448
trt_sh.cgs          36720
trt_oe              34171
trt_sh.css          24368
ctl_vector          14969
trt_lig              7546
ctl_untrt            5332
trt_aby               575
trt_si                162
ctl_vector.cns        137
ctl_vehicle.cns        61
ctl_untrt.cns          30
ctl_x                   1
Name: count, dtype: int64

In [23]:
tar_meta.iloc[0,:]

bead_batch                                                                    b17
nearest_dose                                                                  NaN
pert_dose                                                                   100.0
pert_dose_unit                                                              ug/ml
pert_idose                                                              100 ug/ml
pert_itime                                                                  336 h
pert_time                                                                   336.0
pert_time_unit                                                                  h
cell_mfc_name                                                                  N8
pert_mfc_id                                                         BRD-U44432129
nsample                                                                         4
cc_q75                                                                     0.6164
ss_ngene        

In [24]:
tar_meta_refine = tar_meta.loc[:,['pert_type','cell_iname','cmap_name']].copy() 
#tar_meta_refine.to_csv('/public/home/caojun/project/crane_novel/2_true_data/input/cmap/tar_info.csv')

In [25]:
tar_meta_refine 

,pert_type,cell_iname,cmap_name
0,trt_cp,NAMEC8,BRD-U44432129
1,trt_cp,A549,vorinostat
2,trt_cp,HT29,HMN-214
3,trt_cp,HME1,vorinostat
4,trt_cp,H1975,tanespimycin
...,...,...,...
1201939,trt_cp,HCC515,BRD-K48853221
1201940,trt_cp,HCC515,GW-843682X
1201941,trt_cp,HCC515,BRD-K45785972
1201942,trt_cp,A375,BRD-K28513938


## 1.4 干扰靶点信息

In [26]:
# 干扰靶点信息
ti_meta = pd.read_csv('/public/home/caojun/project/crane_novel/2_true_data/input/cmap/tar_info.csv')

In [27]:
ti_meta

,Unnamed: 0,pert_type,cell_iname,cmap_name
0,0,trt_cp,NAMEC8,BRD-U44432129
1,1,trt_cp,A549,vorinostat
2,2,trt_cp,HT29,HMN-214
3,3,trt_cp,HME1,vorinostat
4,4,trt_cp,H1975,tanespimycin
...,...,...,...,...
1201939,1201939,trt_cp,HCC515,BRD-K48853221
1201940,1201940,trt_cp,HCC515,GW-843682X
1201941,1201941,trt_cp,HCC515,BRD-K45785972
1201942,1201942,trt_cp,A375,BRD-K28513938


In [28]:
ti_meta['pert_type'].value_counts()

pert_type
trt_cp             720216
trt_sh             177263
trt_xpr            140945
ctl_vehicle         39448
trt_sh.cgs          36720
trt_oe              34171
trt_sh.css          24368
ctl_vector          14969
trt_lig              7546
ctl_untrt            5332
trt_aby               575
trt_si                162
ctl_vector.cns        137
ctl_vehicle.cns        61
ctl_untrt.cns          30
ctl_x                   1
Name: count, dtype: int64

## 1.5 化学分子信息

In [29]:
cp_meta = pd.read_csv('/public/home/caojun/project/crane_novel/2_true_data/input/cmap/cp_info_beta.csv')

In [30]:
cp_meta.index = cp_meta['pert_id']

In [31]:
cp_meta.index.name =None

In [32]:
cp_meta 

,pert_id,cmap_name,target,moa,canonical_smiles,inchi_key,compound_aliases
BRD-A08715367,BRD-A08715367,L-theanine,NaN,NaN,CCNC(=O)CCC(N)C(O)=O,DATAGRPVKZEWHA-UHFFFAOYSA-N,l-theanine
BRD-A12237696,BRD-A12237696,L-citrulline,NaN,NaN,NC(CCCNC(N)=O)C(O)=O,RHGKLRLOHDJJDR-UHFFFAOYSA-N,l-citrulline
BRD-A18795974,BRD-A18795974,BRD-A18795974,NaN,NaN,CCCN(CCC)C1CCc2ccc(O)cc2C1,BLYMJBIZMIGWFK-UHFFFAOYSA-N,7-hydroxy-DPAT
BRD-A27924917,BRD-A27924917,BRD-A27924917,NaN,NaN,NCC(O)(CS(O)(=O)=O)c1ccc(Cl)cc1,WBSMZVIMANOCNX-UHFFFAOYSA-N,2-hydroxysaclofen
BRD-A35931254,BRD-A35931254,BRD-A35931254,NaN,NaN,CN1CCc2cccc-3c2C1Cc1ccc(O)c(O)c-31,VMWNQDUVQKEIOC-UHFFFAOYSA-N,r(-)-apomorphine
...,...,...,...,...,...,...,...
BRD-K62685538,BRD-K62685538,triptorelin,GNRHR,Gonadotropin releasing factor hormone receptor...,CC(C)C[C@H](NC(=O)[C@@H](Cc1c[nH]c2ccccc12)NC(...,VXKHXGOKWPXYNA-PGBVPBMZSA-N,NaN
BRD-K62221994,BRD-K62221994,T-98475,GNRHR,Gonadotropin releasing factor hormone receptor...,CC(C)OC(=O)c1cn(Cc2c(F)cccc2F)c3sc(c(CN(C)Cc4c...,RANJJVIMTOIWIN-UHFFFAOYSA-N,NaN
BRD-K53397409,BRD-K53397409,benzoic-acid,RAB9A,"Precursor for food preservatives, plasticizers...",OC(=O)c1ccccc1,WPYMKLBDIGXBTP-UHFFFAOYSA-N,NaN
BRD-A62182663,BRD-A62182663,YK-4279,DHX9,Binding of RNA helicase A to the transcription...,COc1ccc(cc1)C(=O)CC1(O)C(=O)Nc2c1c(Cl)ccc2Cl,HLXSCTYHLQHQDJ-UHFFFAOYSA-N,NaN


In [33]:
cp_meta 

,pert_id,cmap_name,target,moa,canonical_smiles,inchi_key,compound_aliases
BRD-A08715367,BRD-A08715367,L-theanine,NaN,NaN,CCNC(=O)CCC(N)C(O)=O,DATAGRPVKZEWHA-UHFFFAOYSA-N,l-theanine
BRD-A12237696,BRD-A12237696,L-citrulline,NaN,NaN,NC(CCCNC(N)=O)C(O)=O,RHGKLRLOHDJJDR-UHFFFAOYSA-N,l-citrulline
BRD-A18795974,BRD-A18795974,BRD-A18795974,NaN,NaN,CCCN(CCC)C1CCc2ccc(O)cc2C1,BLYMJBIZMIGWFK-UHFFFAOYSA-N,7-hydroxy-DPAT
BRD-A27924917,BRD-A27924917,BRD-A27924917,NaN,NaN,NCC(O)(CS(O)(=O)=O)c1ccc(Cl)cc1,WBSMZVIMANOCNX-UHFFFAOYSA-N,2-hydroxysaclofen
BRD-A35931254,BRD-A35931254,BRD-A35931254,NaN,NaN,CN1CCc2cccc-3c2C1Cc1ccc(O)c(O)c-31,VMWNQDUVQKEIOC-UHFFFAOYSA-N,r(-)-apomorphine
...,...,...,...,...,...,...,...
BRD-K62685538,BRD-K62685538,triptorelin,GNRHR,Gonadotropin releasing factor hormone receptor...,CC(C)C[C@H](NC(=O)[C@@H](Cc1c[nH]c2ccccc12)NC(...,VXKHXGOKWPXYNA-PGBVPBMZSA-N,NaN
BRD-K62221994,BRD-K62221994,T-98475,GNRHR,Gonadotropin releasing factor hormone receptor...,CC(C)OC(=O)c1cn(Cc2c(F)cccc2F)c3sc(c(CN(C)Cc4c...,RANJJVIMTOIWIN-UHFFFAOYSA-N,NaN
BRD-K53397409,BRD-K53397409,benzoic-acid,RAB9A,"Precursor for food preservatives, plasticizers...",OC(=O)c1ccccc1,WPYMKLBDIGXBTP-UHFFFAOYSA-N,NaN
BRD-A62182663,BRD-A62182663,YK-4279,DHX9,Binding of RNA helicase A to the transcription...,COc1ccc(cc1)C(=O)CC1(O)C(=O)Nc2c1c(Cl)ccc2Cl,HLXSCTYHLQHQDJ-UHFFFAOYSA-N,NaN


In [34]:
cp_info_sub = cp_meta.loc[cp_meta['pert_id']=='BRD-K16730910',:]

cp_info_sub

,pert_id,cmap_name,target,moa,canonical_smiles,inchi_key,compound_aliases
BRD-K16730910,BRD-K16730910,regorafenib,FGFR1,KIT inhibitor,CNC(=O)c1cc(Oc2ccc(NC(=O)Nc3ccc(Cl)c(c3)C(F)(F...,FNHKPVJBJVTLMP-UHFFFAOYSA-N,NaN
BRD-K16730910,BRD-K16730910,regorafenib,FGFR2,KIT inhibitor,CNC(=O)c1cc(Oc2ccc(NC(=O)Nc3ccc(Cl)c(c3)C(F)(F...,FNHKPVJBJVTLMP-UHFFFAOYSA-N,NaN
BRD-K16730910,BRD-K16730910,regorafenib,FLT1,KIT inhibitor,CNC(=O)c1cc(Oc2ccc(NC(=O)Nc3ccc(Cl)c(c3)C(F)(F...,FNHKPVJBJVTLMP-UHFFFAOYSA-N,NaN
BRD-K16730910,BRD-K16730910,regorafenib,FLT4,KIT inhibitor,CNC(=O)c1cc(Oc2ccc(NC(=O)Nc3ccc(Cl)c(c3)C(F)(F...,FNHKPVJBJVTLMP-UHFFFAOYSA-N,NaN
BRD-K16730910,BRD-K16730910,regorafenib,FRK,KIT inhibitor,CNC(=O)c1cc(Oc2ccc(NC(=O)Nc3ccc(Cl)c(c3)C(F)(F...,FNHKPVJBJVTLMP-UHFFFAOYSA-N,NaN
...,...,...,...,...,...,...,...
BRD-K16730910,BRD-K16730910,regorafenib,MAPK11,RET tyrosine kinase inhibitor,CNC(=O)c1cc(Oc2ccc(NC(=O)Nc3ccc(Cl)c(c3)C(F)(F...,FNHKPVJBJVTLMP-UHFFFAOYSA-N,NaN
BRD-K16730910,BRD-K16730910,regorafenib,RAF1,RET tyrosine kinase inhibitor,CNC(=O)c1cc(Oc2ccc(NC(=O)Nc3ccc(Cl)c(c3)C(F)(F...,FNHKPVJBJVTLMP-UHFFFAOYSA-N,NaN
BRD-K16730910,BRD-K16730910,regorafenib,RET,RET tyrosine kinase inhibitor,CNC(=O)c1cc(Oc2ccc(NC(=O)Nc3ccc(Cl)c(c3)C(F)(F...,FNHKPVJBJVTLMP-UHFFFAOYSA-N,NaN
BRD-K16730910,BRD-K16730910,regorafenib,BRAF,RET tyrosine kinase inhibitor,CNC(=O)c1cc(Oc2ccc(NC(=O)Nc3ccc(Cl)c(c3)C(F)(F...,FNHKPVJBJVTLMP-UHFFFAOYSA-N,NaN


In [35]:
cp_meta['pert_id'].value_counts()

pert_id
BRD-K16730910    96
BRD-K49810818    66
BRD-K23984367    66
BRD-K29653726    66
BRD-K49328571    60
                 ..
BRD-K65716211     1
BRD-K65729053     1
BRD-K65730939     1
BRD-K65735929     1
BRD-K65655554     1
Name: count, Length: 34419, dtype: int64

In [36]:
unique_drug = list(cp_meta['pert_id'].unique())
len(cp_meta['pert_id'].unique())

34419

In [37]:
`

SyntaxError: invalid syntax (473536216.py, line 1)

In [ ]:
# 整理药物靶点和moa信息
cp_meta_unique = pd.DataFrame(columns=['drug','target','moa','aliases'])
for drug in unique_drug:
    cp_info_sub = cp_meta.loc[cp_meta['pert_id']==drug,:].copy()
    cp_info_sub['target'] = cp_info_sub['target'].fillna('')
    cp_info_sub['moa'] = cp_info_sub['moa'].fillna('') 
    cp_info_sub['compound_aliases'] = cp_info_sub['compound_aliases'].fillna('')
    target_unique = list(cp_info_sub['target'].unique())
    if len(target_unique) > 1:
        target = '; '.join(target_unique)
    else:
        target = target_unique[0]
    moa_unique = list(cp_info_sub['moa'].unique())
    if len(moa_unique) > 1:
        moa = '; '.join(moa_unique)
    else:
        moa = moa_unique[0]
    cmap_name_unique = list(cp_info_sub['cmap_name'].unique())
    compound_aliases_unique = list(cp_info_sub['compound_aliases'].unique())
    aliases_unique = cmap_name_unique + compound_aliases_unique
    if len(aliases_unique) > 1:
        aliases = '; '.join(aliases_unique)
    else:
        aliases = aliases_unique[0]

    #moa = '; '.join(list(cp_info_sub['moa'].unique()))
    #aliases = '; '.join(list(cp_info_sub['cmap_name'].unique())) + '; ' + '; '.join(list(cp_info_sub['compound_aliases'].unique()))
    cp_meta_unique.loc[len(cp_meta_unique)] = {'drug': drug, 'target': target, 'moa': moa, 'aliases': aliases,}

In [37]:
#cp_meta_unique.to_csv('/public/home/caojun/project/RUSH/3_work/input/cp_meta_unique.csv')

In [ ]:
cp_meta_unique

,drug,target,moa,aliases
BRD-A08715367,BRD-A08715367,,,L-theanine; l-theanine
BRD-A12237696,BRD-A12237696,,,L-citrulline; l-citrulline
BRD-A18795974,BRD-A18795974,,,BRD-A18795974; 7-hydroxy-DPAT
BRD-A27924917,BRD-A27924917,,,BRD-A27924917; 2-hydroxysaclofen
BRD-A35931254,BRD-A35931254,,,BRD-A35931254; r(-)-apomorphine
...,...,...,...,...
BRD-K55454768,BRD-K55454768,CAMK2A,Calcium/calmodulin dependent protein kinase in...,TAS-301;
BRD-K99504665,BRD-K99504665,GNRHR,Gonadotropin releasing factor hormone receptor...,goserelin-acetate;
BRD-K62685538,BRD-K62685538,GNRHR,Gonadotropin releasing factor hormone receptor...,triptorelin;
BRD-K62221994,BRD-K62221994,GNRHR,Gonadotropin releasing factor hormone receptor...,T-98475;


# 2. gctx

In [ ]:
# /public/home/caojun/project/crane_novel/2_true_data/input/cmap

In [ ]:
# https://clue.io/releases/data-dashboard


# Level 5 replicate-collapsed z-score vectors based on Level 4. 
# Level 4 - ZS：基于Level 3的数据，计算每个基因的Z分数。通过将样本与同一板块的对照组进行比较，识别出差异表达的基因。

# Level 5 - MODZ：在这一层，基于Level 4生成的重复品合并Z分数向量，简化为一个称为“特征”的差异表达向量。
#大多数连接性分析都是在这一层的数据上进行。

# 对照组
# level5_beta_ctl_n58022x12328.gctx	58022x12328	2.66 GB	4c70a6939637670d185775f2a2d98d67

# Compound treatment data 
# level5_beta_trt_cp_n720216x12328.gctx	720216x12328	33.08 GB	9a82806e2aba6ec2a866cba77bd57fda

# 杂项 treatment data 
# level5_beta_trt_misc_n8283x12328.gctx	8283x12328	389.61 MB	b58bcaa628f9f2afeadbb815b49a7684

# 基因过表达 treatment data
# level5_beta_trt_oe_n34171x12328.gctx	34171x12328	1.57 GB	a068e259e2d71676e8fa46a2d7a2de86

# shRNA处理 treatment data
# level5_beta_trt_sh_n238351x12328.gctx	238351x12328	10.95 GB	16952edbdc39756370a075b25f874029


# CRISPR for LoF treatment data
# level5_beta_trt_xpr_n142901x12328.gctx	142901x12328	6.07 GB	c852ca26affaa144f1b042463036702b
 

## 2.1 cp

### 2.1.1 all

In [21]:
path_to_gctx = "/public/home/caojun/project/crane_novel/2_true_data/input/cmap/level5_beta_trt_cp_n720216x12328.gctx"  
cp_gctx_data = parse(path_to_gctx)  

/public/home/caojun/anaconda3/envs/crane_test/lib/python3.10/site-packages/cmapPy/pandasGEXpress/parse_gctx.py:275: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  meta_df = meta_df.apply(lambda x: pd.to_numeric(x, errors="ignore"))
/public/home/caojun/anaconda3/envs/crane_test/lib/python3.10/site-packages/cmapPy/pandasGEXpress/parse_gctx.py:275: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  meta_df = meta_df.apply(lambda x: pd.to_numeric(x, errors="ignore"))


In [24]:
cp_gctx_data.data_df.shape

(12328, 720216)

In [32]:
cp_mat = cp_gctx_data.data_df
cp_adata = ad.AnnData(cp_mat.T)
cp_adata.var = gene_meta.loc[cp_mat.index,:]
tar_cp_meta = tar_meta.loc[tar_meta['pert_type']=='trt_cp',:]
tar_cp_meta.index = tar_cp_meta['sig_id'].values.copy()
cp_adata.obs = tar_cp_meta.loc[cp_adata.obs_names,:].copy()

In [54]:
cp_adata

AnnData object with n_obs × n_vars = 720216 × 12328
    obs: 'bead_batch', 'nearest_dose', 'pert_dose', 'pert_dose_unit', 'pert_idose', 'pert_itime', 'pert_time', 'pert_time_unit', 'cell_mfc_name', 'pert_mfc_id', 'nsample', 'cc_q75', 'ss_ngene', 'tas', 'pct_self_rank_q25', 'wt', 'median_recall_rank_spearman', 'median_recall_rank_wtcs_50', 'median_recall_score_spearman', 'median_recall_score_wtcs_50', 'batch_effect_tstat', 'batch_effect_tstat_pct', 'is_hiq', 'qc_pass', 'pert_id', 'sig_id', 'pert_type', 'cell_iname', 'det_wells', 'det_plates', 'distil_ids', 'build_name', 'project_code', 'cmap_name', 'is_exemplar_sig', 'is_ncs_sig', 'is_null_sig'
    var: 'gene_id', 'gene_symbol', 'ensembl_id', 'gene_title', 'gene_type', 'src', 'feature_space'

In [55]:
cp_adata.write('/public/home/caojun/project/RUSH/3_work/input/cmap_cp_ad.h5ad')

### 2.1.2 unique

In [ ]:
# 每个药只保留一个代表！！！

In [104]:
#cp_meta_unique.to_csv('/public/home/caojun/project/RUSH/3_work/input/cp_meta_unique.csv')
cp_meta_unique.index = cp_meta_unique['drug'].values.copy()

In [105]:
cp_meta_unique

,drug,target,moa,aliases
BRD-A08715367,BRD-A08715367,,,L-theanine; l-theanine
BRD-A12237696,BRD-A12237696,,,L-citrulline; l-citrulline
BRD-A18795974,BRD-A18795974,,,BRD-A18795974; 7-hydroxy-DPAT
BRD-A27924917,BRD-A27924917,,,BRD-A27924917; 2-hydroxysaclofen
BRD-A35931254,BRD-A35931254,,,BRD-A35931254; r(-)-apomorphine
...,...,...,...,...
BRD-K55454768,BRD-K55454768,CAMK2A,Calcium/calmodulin dependent protein kinase in...,TAS-301;
BRD-K99504665,BRD-K99504665,GNRHR,Gonadotropin releasing factor hormone receptor...,goserelin-acetate;
BRD-K62685538,BRD-K62685538,GNRHR,Gonadotropin releasing factor hormone receptor...,triptorelin;
BRD-K62221994,BRD-K62221994,GNRHR,Gonadotropin releasing factor hormone receptor...,T-98475;


In [58]:
cp_adata.obs = cp_adata.obs.loc[:,['sig_id','project_code','det_plates','cell_iname','pert_id','cmap_name','pert_dose','pert_itime']]

In [107]:
pd.crosstab(
    cp_adata.obs['cell_iname'],
    cp_adata.obs['pert_id']
)

pert_id,943,949,BRD2492 + BRD3308,BRD-A00077618,BRD-A00100033,BRD-A00147595,BRD-A00150179,BRD-A00218260,BRD-A00267231,BRD-A00327403,...,LIVF001-027,LIVF001-028,LIVF001-029,LUC_ALARIN,LUC_FBS,LUC_GALANIN,LUC_GALP,LUC_M617,SAHA,SRT3657
cell_iname,,,,,,,,,,,,,,,,,,,,,
1HAE,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
22RV1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5637,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
A204,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
A375,0,0,0,3,1,12,0,12,1,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
XC.P936,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
XC.R10,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
YAPC,0,0,0,0,0,12,0,12,0,0,...,0,0,0,0,0,0,0,0,0,0


In [178]:
cp_adata.obs.loc[(cp_adata.obs['pert_id']=='BRD-A00100033') & (cp_adata.obs['cell_iname']=='A375'),:]

,sig_id,project_code,det_plates,cell_iname,pert_id,cmap_name,pert_dose,pert_itime,hit
cid,,,,,,,,,
CPC015_A375_6H:BRD-A00100033-001-04-8:10,CPC015_A375_6H:BRD-A00100033-001-04-8:10,CPC,CPC015_A375_6H_X1_B4_DUO52HI53LO|CPC015_A375_6...,A375,BRD-A00100033,nifurtimox,10.0,6.0,True


In [59]:
cp_adata.obs

,sig_id,project_code,det_plates,cell_iname,pert_id,cmap_name,pert_dose,pert_itime
cid,,,,,,,,
ABY001_A375_XH:BRD-A61304759:0.625:24,ABY001_A375_XH:BRD-A61304759:0.625:24,ABY,ABY001_A375_XH_X1_B15,A375,BRD-A61304759,tanespimycin,0.625,24 h
ABY001_A375_XH:BRD-A61304759:0.625:3,ABY001_A375_XH:BRD-A61304759:0.625:3,ABY,ABY001_A375_XH_X1_B15,A375,BRD-A61304759,tanespimycin,0.625,3 h
ABY001_A375_XH:BRD-A61304759:10:24,ABY001_A375_XH:BRD-A61304759:10:24,ABY,ABY001_A375_XH_X1_B15,A375,BRD-A61304759,tanespimycin,10.000,24 h
ABY001_A375_XH:BRD-A61304759:10:3,ABY001_A375_XH:BRD-A61304759:10:3,ABY,ABY001_A375_XH_X1_B15,A375,BRD-A61304759,tanespimycin,10.000,3 h
ABY001_A375_XH:BRD-A61304759:2.5:24,ABY001_A375_XH:BRD-A61304759:2.5:24,ABY,ABY001_A375_XH_X1_B15,A375,BRD-A61304759,tanespimycin,2.500,24 h
...,...,...,...,...,...,...,...,...
TSAI002_NPC-8_XH:CI-994:10,TSAI002_NPC-8_XH:CI-994:10,TSAI,TSAI002_NPC-8_XH_X1_B18,NPC,CI-994,CI-994,10.000,NaN
TSAI002_NPC-8_XH:COMPE:2,TSAI002_NPC-8_XH:COMPE:2,TSAI,TSAI002_NPC-8_XH_X1_B18,NPC,COMPE,compe,2.000,NaN
TSAI002_NPC-8_XH:DAC-3:5,TSAI002_NPC-8_XH:DAC-3:5,TSAI,TSAI002_NPC-8_XH_X1_B18,NPC,DAC-3,DAC-3,5.000,NaN


In [ ]:
cp_adata.obs['pert_itime'] = cp_adata.obs['pert_itime'].str.extract(r'([\d.]+)', expand=False).fillna(0).astype(float)

In [ ]:
import numpy as np

# 将 pert_dose 和 pert_itime 转换为数值型数据，提取最大剂量和最大时间
#cp_adata.obs['pert_dose'] = cp_adata.obs['pert_dose'].str.extract(r'([\d.]+)', expand=False).fillna(0).astype(float)

# 通过 groupby 来按细胞系和药物分组
def get_max_values(group):
    max_dose = group['pert_dose'].max()
    max_itime = group['pert_itime'].max()
    
    # 找到最大剂量和最大时间的索引，确保有符合条件的行
    max_row = group[(group['pert_dose'] == max_dose) & (group['pert_itime'] == max_itime)]
    
    if not max_row.empty:  # 确保 max_row 非空
        return max_row.index[0]  # 返回第一行的索引（即代表值）
    else:
        return np.nan  # 如果没有符合条件的行，返回 NaN

# 获取每个分组的代表值
representative_ids = cp_adata.obs.groupby(['pert_id', 'cell_iname']).apply(get_max_values)

# 删除 NaN 行，因为它们没有代表值
representative_ids = representative_ids.dropna()

# 将代表值的 hit 标记为 True
cp_adata.obs['hit'] = cp_adata.obs.index.isin(representative_ids)


/tmp/ipykernel_28954/2634238035.py:20: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  representative_ids = cp_adata.obs.groupby(['pert_id', 'cell_iname']).apply(get_max_values)
/tmp/ipykernel_28954/2634238035.py:20: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  representative_ids = cp_adata.obs.groupby(['pert_id', 'cell_iname']).apply(get_max_values)


In [173]:
cp_adata.obs['hit'].value_counts()

hit
False    519202
True     201014
Name: count, dtype: int64

In [175]:
cp_adata[cp_adata.obs['hit'],:]

View of AnnData object with n_obs × n_vars = 201014 × 12328
    obs: 'sig_id', 'project_code', 'det_plates', 'cell_iname', 'pert_id', 'cmap_name', 'pert_dose', 'pert_itime', 'hit'
    var: 'gene_id', 'gene_symbol', 'ensembl_id', 'gene_title', 'gene_type', 'src', 'feature_space'

In [174]:
cp_adata[cp_adata.obs['hit'],:].write('/public/home/caojun/project/RUSH/3_work/input/cmap_cp_ad_unique.h5ad')

### 2.2.3 split

In [39]:
cp_adata = sc.read_h5ad('/public/home/caojun/project/RUSH/3_work/input/cmap_cp_ad_unique.h5ad')

In [40]:
cp_adata

AnnData object with n_obs × n_vars = 201014 × 12328
    obs: 'sig_id', 'project_code', 'det_plates', 'cell_iname', 'pert_id', 'cmap_name', 'pert_dose', 'pert_itime', 'hit'
    var: 'gene_id', 'gene_symbol', 'ensembl_id', 'gene_title', 'gene_type', 'src', 'feature_space'

In [41]:
output_dir = "/public/home/caojun/project/RUSH/3_work/output/store/cp_split"

In [42]:
# 确定每个文件保存多少行数据
chunk_size = 1000

# 计算切分的数量
n_chunks = np.ceil(len(cp_adata.obs) / chunk_size).astype(int)

# 切分并保存
for i in range(n_chunks):
    # 切分数据，取第 i 个区间的 1000 行
    start_idx = i * chunk_size
    end_idx = min((i + 1) * chunk_size, len(cp_adata.obs))
    
    # 创建一个新的 AnnData 子集
    cp_adata_sub = cp_adata[start_idx:end_idx].copy()
    
    # 保存为 .h5ad 文件
    file_name = f"cp_split_{i}.h5ad"
    file_path = os.path.join(output_dir, file_name)
    
    # 写入文件
    cp_adata_sub.write(file_path)

    print(f"Saved {file_name} to {file_path}")


Saved cp_split_0.h5ad to /public/home/caojun/project/RUSH/3_work/output/store/cp_split/cp_split_0.h5ad
Saved cp_split_1.h5ad to /public/home/caojun/project/RUSH/3_work/output/store/cp_split/cp_split_1.h5ad
Saved cp_split_2.h5ad to /public/home/caojun/project/RUSH/3_work/output/store/cp_split/cp_split_2.h5ad
Saved cp_split_3.h5ad to /public/home/caojun/project/RUSH/3_work/output/store/cp_split/cp_split_3.h5ad
Saved cp_split_4.h5ad to /public/home/caojun/project/RUSH/3_work/output/store/cp_split/cp_split_4.h5ad
Saved cp_split_5.h5ad to /public/home/caojun/project/RUSH/3_work/output/store/cp_split/cp_split_5.h5ad
Saved cp_split_6.h5ad to /public/home/caojun/project/RUSH/3_work/output/store/cp_split/cp_split_6.h5ad
Saved cp_split_7.h5ad to /public/home/caojun/project/RUSH/3_work/output/store/cp_split/cp_split_7.h5ad
Saved cp_split_8.h5ad to /public/home/caojun/project/RUSH/3_work/output/store/cp_split/cp_split_8.h5ad
Saved cp_split_9.h5ad to /public/home/caojun/project/RUSH/3_work/output/s

## 2.2 shrna

### 2.2.1 all

In [45]:
path_to_gctx = "/public/home/caojun/project/crane_novel/2_true_data/input/cmap/level5_beta_trt_sh_n238351x12328.gctx"  
sh_gctx_data = parse(path_to_gctx)  

/public/home/caojun/anaconda3/envs/crane_test/lib/python3.10/site-packages/cmapPy/pandasGEXpress/parse_gctx.py:275: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  meta_df = meta_df.apply(lambda x: pd.to_numeric(x, errors="ignore"))
/public/home/caojun/anaconda3/envs/crane_test/lib/python3.10/site-packages/cmapPy/pandasGEXpress/parse_gctx.py:275: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  meta_df = meta_df.apply(lambda x: pd.to_numeric(x, errors="ignore"))


In [46]:
sh_gctx_data.data_df.shape

(12328, 238351)

In [51]:
sh_mat = sh_gctx_data.data_df
sh_adata = ad.AnnData(sh_mat.T)
sh_adata.var = gene_meta.loc[sh_mat.index,:]

In [64]:
tar_meta['pert_type'].value_counts()

pert_type
trt_cp             720216
trt_sh             177263
trt_xpr            140945
ctl_vehicle         39448
trt_sh.cgs          36720
trt_oe              34171
trt_sh.css          24368
ctl_vector          14969
trt_lig              7546
ctl_untrt            5332
trt_aby               575
trt_si                162
ctl_vector.cns        137
ctl_vehicle.cns        61
ctl_untrt.cns          30
ctl_x                   1
Name: count, dtype: int64

In [66]:
tar_sh_meta = tar_meta.loc[tar_meta['pert_type'].isin(['trt_sh','trt_sh.cgs','trt_sh.css']),:]

In [54]:
sh_mat

cid,CGS001_A375_96H:A2M:1,CGS001_A375_96H:AARS:1,CGS001_A375_96H:AATF:1,CGS001_A375_96H:ABAT:1,CGS001_A375_96H:ABCA1:1,CGS001_A375_96H:ABCA3:1,CGS001_A375_96H:ABCA5:1,CGS001_A375_96H:ABCB1:1,CGS001_A375_96H:ABCB4:1,CGS001_A375_96H:ABCB6:1,...,TAK004_U2OS_96H:TRCN0000364646:1,TAK004_U2OS_96H:TRCN0000364647:1,TAK004_U2OS_96H:TRCN0000369292:1,TAK004_U2OS_96H:TRCN0000369366:1,TAK004_U2OS_96H:TRCN0000370006:1,TAK004_U2OS_96H:TRCN0000370007:1,TAK004_U2OS_96H:TRCN0000370678:1,TAK004_U2OS_96H:TRCN0000370697:1,TAK004_U2OS_96H:TRCN0000370751:1,TAK004_U2OS_96H:TRCN0000381509:1
rid,,,,,,,,,,,,,,,,,,,,,
10,-0.301409,-0.179307,-0.354759,0.168650,-0.104921,-0.221035,0.025308,0.525450,-0.399560,0.108029,...,-0.917287,0.318601,1.315510,1.140851,0.047501,0.330383,0.020266,-0.061204,0.858136,0.011221
100,0.291780,0.052339,0.696610,-0.421365,-0.197109,-0.114712,0.669819,-0.153325,-0.130811,-0.475240,...,-1.054414,0.475651,0.145515,0.491073,1.368213,-0.147739,-0.495189,-1.065773,0.346499,0.566997
1000,-0.175148,0.616403,1.242735,-0.499108,0.074730,0.730160,0.480573,-0.083900,1.060225,0.387124,...,0.372930,0.102032,-0.317871,-0.199936,-1.134715,0.692574,-0.929272,0.379051,-0.770353,0.178147
10000,-0.510091,0.191684,0.296094,-0.830489,0.648741,-0.403104,-0.041393,0.473325,-0.588845,-0.042834,...,-0.563395,1.327857,0.412632,0.865327,-1.320900,0.653349,0.132837,0.351420,-0.639036,-1.110442
10001,0.131631,0.153195,0.596103,-0.936329,-0.052865,1.300483,0.759824,-0.207725,0.928339,0.056673,...,1.177266,-1.205259,-0.679170,-0.964915,-0.773761,0.596456,0.145665,-0.223776,0.157762,-0.052088
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9990,-0.209862,-0.612235,-0.272712,0.220896,0.571486,0.517760,-0.838253,0.371325,-1.050902,0.318720,...,-0.101760,0.860102,0.120347,-0.357495,-0.511063,0.688053,0.045342,-0.380661,1.409096,0.651931
9991,0.250892,0.392434,0.990092,-0.108615,0.427193,0.338683,0.628986,0.738475,-0.781355,0.561399,...,-0.531004,-0.411433,0.008842,-0.616618,-0.220280,-0.067348,-0.044710,-0.436087,-0.440849,-0.195389
9992,-0.348875,-0.259771,0.764011,0.338886,0.521197,-0.483014,-0.203921,0.961225,-1.461934,-0.662574,...,-0.898100,1.385529,0.819197,0.503125,0.977125,-0.373416,0.839890,-0.275064,0.709834,-0.182312


In [69]:
tar_sh_meta.index = tar_sh_meta['sig_id'].values.copy()
sh_adata.obs = tar_sh_meta.loc[sh_adata.obs_names,:].copy()

In [70]:
sh_adata

AnnData object with n_obs × n_vars = 238351 × 12328
    obs: 'bead_batch', 'nearest_dose', 'pert_dose', 'pert_dose_unit', 'pert_idose', 'pert_itime', 'pert_time', 'pert_time_unit', 'cell_mfc_name', 'pert_mfc_id', 'nsample', 'cc_q75', 'ss_ngene', 'tas', 'pct_self_rank_q25', 'wt', 'median_recall_rank_spearman', 'median_recall_rank_wtcs_50', 'median_recall_score_spearman', 'median_recall_score_wtcs_50', 'batch_effect_tstat', 'batch_effect_tstat_pct', 'is_hiq', 'qc_pass', 'pert_id', 'sig_id', 'pert_type', 'cell_iname', 'det_wells', 'det_plates', 'distil_ids', 'build_name', 'project_code', 'cmap_name', 'is_exemplar_sig', 'is_ncs_sig', 'is_null_sig'
    var: 'gene_id', 'gene_symbol', 'ensembl_id', 'gene_title', 'gene_type', 'src', 'feature_space'

In [71]:
sh_adata.write('/public/home/caojun/project/RUSH/3_work/input/cmap_sh_ad.h5ad')

### 2.2.2 unique

In [ ]:
# 每个扰动只保留一个代表！！！

In [79]:
tar_sh_meta_sub = tar_sh_meta.loc[:,['sig_id','project_code','cell_iname','pert_id','cmap_name', 'pert_dose', 'pert_time']]

In [89]:
sh666 = tar_sh_meta_sub.index[tar_sh_meta_sub['pert_dose']==-666.0].to_list()

In [91]:
tar_sh_meta_sub = tar_sh_meta_sub.loc[~tar_sh_meta_sub.index.isin(sh666),:]

In [92]:
tar_sh_meta_sub

,sig_id,project_code,cell_iname,pert_id,cmap_name,pert_dose,pert_time
TAK001_PC3_96H:TRCN0000350275:-666,TAK001_PC3_96H:TRCN0000350275:-666,TAK,PC3,TRCN0000350275,PIK3CA,NaN,96.0
TAK001_PC3_96H:TRCN0000001521:-666,TAK001_PC3_96H:TRCN0000001521:-666,TAK,PC3,TRCN0000001521,SGK3,NaN,96.0
TAK004_U2OS_96H:TRCN0000255441:1,TAK004_U2OS_96H:TRCN0000255441:1,TAK,U2OS,TRCN0000255441,LMNB2,1.0,96.0
TAK001_HEKTE_96H:TRCN0000058503:-666,TAK001_HEKTE_96H:TRCN0000058503:-666,TAK,HEKTE,TRCN0000058503,VEGFC,NaN,96.0
DER001_HA1E_96H:TRCN0000195187:-666,DER001_HA1E_96H:TRCN0000195187:-666,DER,HA1E,TRCN0000195187,RIOK3,NaN,96.0
...,...,...,...,...,...,...,...
ERGK005_VCAP_120H:TRCN0000006197:-666,ERGK005_VCAP_120H:TRCN0000006197:-666,ERGK,VCAP,TRCN0000006197,PHKB,NaN,120.0
ERGK005_VCAP_120H:TRCN0000196810:-666,ERGK005_VCAP_120H:TRCN0000196810:-666,ERGK,VCAP,TRCN0000196810,PGK1,NaN,120.0
ERGK002_VCAP_120H:TRCN0000000563:-666,ERGK002_VCAP_120H:TRCN0000000563:-666,ERGK,VCAP,TRCN0000000563,AKT2,NaN,120.0
ERGK011_VCAP_120H:TRCN0000196976:-666,ERGK011_VCAP_120H:TRCN0000196976:-666,ERGK,VCAP,TRCN0000196976,TRCN0000196976,NaN,120.0


In [93]:
tar_sh_meta_sub['pert_dose'].value_counts()

pert_dose
2.0     20510
1.0     19512
1.5     13742
5.0      7065
4.0      2034
6.0      1051
10.0      399
3.0        85
20.0       24
25.0       23
Name: count, dtype: int64

In [94]:
tar_sh_meta_sub.loc[(tar_sh_meta_sub['pert_id']=='CGS001-18') & (tar_sh_meta_sub['cell_iname']=='MCF7'),:]

,sig_id,project_code,cell_iname,pert_id,cmap_name,pert_dose,pert_time
CGS001_MCF7_96H:ABAT:2,CGS001_MCF7_96H:ABAT:2,CGS,MCF7,CGS001-18,ABAT,2.0,96.0
CGS001_MCF7_144H:ABAT:2,CGS001_MCF7_144H:ABAT:2,CGS,MCF7,CGS001-18,ABAT,2.0,144.0


In [95]:
pd.crosstab(
    sh_adata.obs['cell_iname'],
    sh_adata.obs['pert_id']
)

pert_id,CGS001-2,CGS001-9,CGS001-16,CGS001-18,CGS001-19,CGS001-21,CGS001-22,CGS001-23,CGS001-25,CGS001-27,...,TRCN0000552092,TRCN0000552093,TRCN0000552095,TRCN0000552096,TRCN0000552098,TRCN0000552099,TRCN0000552101,TRCN0000552102,TRCN0000552104,TRCN0000552105
cell_iname,,,,,,,,,,,,,,,,,,,,,
A375,1,1,1,1,1,1,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0
A549,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
ASC,0,0,1,1,0,0,0,1,1,0,...,0,0,0,0,0,0,0,0,0,0
DLD1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
HA1E,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
HCC515,0,1,1,1,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
HEK293T,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
HEKTE,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
HEPG2,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0


In [96]:
sh_adata = sh_adata[~sh_adata.obs_names.isin(sh666),:]

In [100]:
sh_adata.obs['pert_itime'] = sh_adata.obs['pert_itime'].fillna(0).astype(float)

In [99]:
sh_adata.obs['pert_dose'] = sh_adata.obs['pert_dose'].fillna(0).astype(float)

In [101]:
import numpy as np

# 将 pert_dose 和 pert_itime 转换为数值型数据，提取最大剂量和最大时间
#cp_adata.obs['pert_dose'] = cp_adata.obs['pert_dose'].str.extract(r'([\d.]+)', expand=False).fillna(0).astype(float)

# 通过 groupby 来按细胞系和药物分组
def get_max_values(group):
    max_dose = group['pert_dose'].max()
    max_itime = group['pert_itime'].max()
    
    # 找到最大剂量和最大时间的索引，确保有符合条件的行
    max_row = group[(group['pert_dose'] == max_dose) & (group['pert_itime'] == max_itime)]
    
    if not max_row.empty:  # 确保 max_row 非空
        return max_row.index[0]  # 返回第一行的索引（即代表值）
    else:
        return np.nan  # 如果没有符合条件的行，返回 NaN

# 获取每个分组的代表值
representative_ids = sh_adata.obs.groupby(['pert_id', 'cell_iname']).apply(get_max_values)

# 删除 NaN 行，因为它们没有代表值
representative_ids = representative_ids.dropna()

# 将代表值的 hit 标记为 True
sh_adata.obs['hit'] = sh_adata.obs.index.isin(representative_ids)


/tmp/ipykernel_19209/2568453184.py:20: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  representative_ids = sh_adata.obs.groupby(['pert_id', 'cell_iname']).apply(get_max_values)
/tmp/ipykernel_19209/2568453184.py:20: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  representative_ids = sh_adata.obs.groupby(['pert_id', 'cell_iname']).apply(get_max_values)


In [102]:
sh_adata.obs['hit'].value_counts()

hit
True     189365
False     48222
Name: count, dtype: int64

In [103]:
sh_adata[sh_adata.obs['hit'],:]

View of AnnData object with n_obs × n_vars = 189365 × 12328
    obs: 'bead_batch', 'nearest_dose', 'pert_dose', 'pert_dose_unit', 'pert_idose', 'pert_itime', 'pert_time', 'pert_time_unit', 'cell_mfc_name', 'pert_mfc_id', 'nsample', 'cc_q75', 'ss_ngene', 'tas', 'pct_self_rank_q25', 'wt', 'median_recall_rank_spearman', 'median_recall_rank_wtcs_50', 'median_recall_score_spearman', 'median_recall_score_wtcs_50', 'batch_effect_tstat', 'batch_effect_tstat_pct', 'is_hiq', 'qc_pass', 'pert_id', 'sig_id', 'pert_type', 'cell_iname', 'det_wells', 'det_plates', 'distil_ids', 'build_name', 'project_code', 'cmap_name', 'is_exemplar_sig', 'is_ncs_sig', 'is_null_sig', 'hit'
    var: 'gene_id', 'gene_symbol', 'ensembl_id', 'gene_title', 'gene_type', 'src', 'feature_space'

In [104]:
sh_adata[sh_adata.obs['hit'],:].write('/public/home/caojun/project/RUSH/3_work/input/cmap_sh_ad_unique.h5ad')

### 2.2.3 split

In [105]:
sh_adata = sc.read_h5ad('/public/home/caojun/project/RUSH/3_work/input/cmap_sh_ad_unique.h5ad')

In [106]:
sh_adata

AnnData object with n_obs × n_vars = 189365 × 12328
    obs: 'bead_batch', 'nearest_dose', 'pert_dose', 'pert_dose_unit', 'pert_idose', 'pert_itime', 'pert_time', 'pert_time_unit', 'cell_mfc_name', 'pert_mfc_id', 'nsample', 'cc_q75', 'ss_ngene', 'tas', 'pct_self_rank_q25', 'wt', 'median_recall_rank_spearman', 'median_recall_rank_wtcs_50', 'median_recall_score_spearman', 'median_recall_score_wtcs_50', 'batch_effect_tstat', 'batch_effect_tstat_pct', 'is_hiq', 'qc_pass', 'pert_id', 'sig_id', 'pert_type', 'cell_iname', 'det_wells', 'det_plates', 'distil_ids', 'build_name', 'project_code', 'cmap_name', 'is_exemplar_sig', 'is_ncs_sig', 'is_null_sig', 'hit'
    var: 'gene_id', 'gene_symbol', 'ensembl_id', 'gene_title', 'gene_type', 'src', 'feature_space'

In [107]:
output_dir = "/public/home/caojun/project/RUSH/3_work/output/store/sh_split"

In [109]:
# 确定每个文件保存多少行数据
chunk_size = 1000

# 计算切分的数量
n_chunks = np.ceil(len(sh_adata.obs) / chunk_size).astype(int)

# 切分并保存
for i in range(n_chunks):
    # 切分数据，取第 i 个区间的 1000 行
    start_idx = i * chunk_size
    end_idx = min((i + 1) * chunk_size, len(sh_adata.obs))
    
    # 创建一个新的 AnnData 子集
    sh_adata_sub = sh_adata[start_idx:end_idx].copy()
    
    # 保存为 .h5ad 文件
    file_name = f"sh_split_{i}.h5ad"
    file_path = os.path.join(output_dir, file_name)
    
    # 写入文件
    sh_adata_sub.write(file_path)

    print(f"Saved {file_name} to {file_path}")


Saved sh_split_0.h5ad to /public/home/caojun/project/RUSH/3_work/output/store/sh_split/sh_split_0.h5ad
Saved sh_split_1.h5ad to /public/home/caojun/project/RUSH/3_work/output/store/sh_split/sh_split_1.h5ad
Saved sh_split_2.h5ad to /public/home/caojun/project/RUSH/3_work/output/store/sh_split/sh_split_2.h5ad
Saved sh_split_3.h5ad to /public/home/caojun/project/RUSH/3_work/output/store/sh_split/sh_split_3.h5ad
Saved sh_split_4.h5ad to /public/home/caojun/project/RUSH/3_work/output/store/sh_split/sh_split_4.h5ad
Saved sh_split_5.h5ad to /public/home/caojun/project/RUSH/3_work/output/store/sh_split/sh_split_5.h5ad
Saved sh_split_6.h5ad to /public/home/caojun/project/RUSH/3_work/output/store/sh_split/sh_split_6.h5ad
Saved sh_split_7.h5ad to /public/home/caojun/project/RUSH/3_work/output/store/sh_split/sh_split_7.h5ad
Saved sh_split_8.h5ad to /public/home/caojun/project/RUSH/3_work/output/store/sh_split/sh_split_8.h5ad
Saved sh_split_9.h5ad to /public/home/caojun/project/RUSH/3_work/output/s

## 2.3 crispr

### 2.3.1 all

In [44]:
path_to_gctx = "/public/home/caojun/project/crane_novel/2_true_data/input/cmap/level5_beta_trt_xpr_n142901x12328.gctx"  
xpr_gctx_data = parse(path_to_gctx)  

/public/home/caojun/anaconda3/envs/crane_test/lib/python3.10/site-packages/cmapPy/pandasGEXpress/parse_gctx.py:275: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  meta_df = meta_df.apply(lambda x: pd.to_numeric(x, errors="ignore"))
/public/home/caojun/anaconda3/envs/crane_test/lib/python3.10/site-packages/cmapPy/pandasGEXpress/parse_gctx.py:275: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  meta_df = meta_df.apply(lambda x: pd.to_numeric(x, errors="ignore"))


In [45]:
xpr_gctx_data.data_df.shape

(12328, 142901)

In [55]:
xpr_mat = xpr_gctx_data.data_df
xpr_adata = ad.AnnData(xpr_mat.T)
xpr_adata.var = gene_meta.loc[xpr_mat.index,:]

In [ ]:
tar_meta['pert_type'].value_counts()

pert_type
trt_cp             720216
trt_sh             177263
trt_xpr            140945
ctl_vehicle         39448
trt_sh.cgs          36720
trt_oe              34171
trt_sh.css          24368
ctl_vector          14969
trt_lig              7546
ctl_untrt            5332
trt_aby               575
trt_si                162
ctl_vector.cns        137
ctl_vehicle.cns        61
ctl_untrt.cns          30
ctl_x                   1
Name: count, dtype: int64

In [54]:
tar_xpr_meta = tar_meta.loc[tar_meta['pert_type'].isin(['trt_xpr', ]),:]

In [ ]:
exit_d = list(set(tar_xpr_meta.index) & set(xpr_mat.columns))
len(exit_d )

140945

In [57]:
xpr_adata = xpr_adata[exit_d,:]

In [48]:
xpr_mat

cid,XPR038_A549.311_96H:A03,XPR038_A549.311_96H:A04,XPR038_A549.311_96H:A05,XPR038_A549.311_96H:A06,XPR038_A549.311_96H:A07,XPR038_A549.311_96H:A08,XPR038_A549.311_96H:A09,XPR038_A549.311_96H:A10,XPR038_A549.311_96H:A11,XPR038_A549.311_96H:A12,...,ZTO.XPR001_THP1_408H:RAD21:-666,ZTO.XPR001_THP1_408H:SMC1A:-666,ZTO.XPR001_THP1_408H:SMC3:-666,ZTO.XPR001_THP1_408H:STAG2:-666,ZTO.XPR001_U937_408H:NIPBL:-666,ZTO.XPR001_U937_408H:PDS5B:-666,ZTO.XPR001_U937_408H:RAD21:-666,ZTO.XPR001_U937_408H:SMC1A:-666,ZTO.XPR001_U937_408H:SMC3:-666,ZTO.XPR001_U937_408H:STAG2:-666
rid,,,,,,,,,,,,,,,,,,,,,
780,-0.397990,0.142264,0.110257,0.390068,0.164016,1.007527,0.653676,-0.213736,0.308933,-0.261417,...,1.200280,1.607262,2.472895,4.091119,-0.189182,-0.004768,0.119435,0.433973,0.205687,0.068893
7849,-1.161871,-0.361102,0.366890,0.058730,-0.169961,1.013682,0.732813,0.003162,0.184261,0.636178,...,-0.479713,0.608731,-0.639864,0.252356,0.330040,-0.387401,-0.052151,-0.155700,-0.302380,-0.015722
2978,-0.277472,-1.190112,-0.285301,-0.528516,0.411186,0.616334,1.233061,-0.843386,1.354372,0.759188,...,-0.083086,-0.152003,0.325006,-0.419212,-0.075683,-0.074463,0.034490,0.283306,-0.324167,0.146801
2049,-0.908711,-0.304021,-1.187883,-0.028075,0.402940,0.513586,-0.474156,-0.438747,1.058755,0.112944,...,-0.117739,-0.211641,0.273923,-0.778768,0.000653,0.292352,0.462498,0.385998,0.153100,-0.607215
2101,-0.200055,0.504097,-0.003579,-0.522273,0.177294,-0.002214,0.672511,-0.707257,0.031196,1.039789,...,-0.121755,0.193579,0.297540,-0.174075,-0.716490,0.118912,0.080506,0.122677,0.374604,0.430878
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4034,-0.162468,0.194514,-0.181924,-1.069134,-0.529307,-0.552736,-0.187219,-0.138709,-0.312385,-0.087262,...,-0.111280,-0.310382,-0.749163,-0.204215,0.000876,-0.407064,-0.340172,-0.230139,-0.123742,0.496653
399664,-0.795937,-0.067918,-0.021914,0.171901,0.045210,0.699309,0.551582,-0.106711,0.880625,1.064388,...,0.322382,0.217394,0.495132,-0.017505,-0.310500,-0.188391,-0.130577,-0.464519,0.016345,0.477860
54869,0.206143,-1.534017,-0.406886,-1.213087,0.592331,-0.036806,0.627547,-1.180923,1.072805,-0.003329,...,-0.504058,0.004845,0.327697,-0.295596,-0.095773,-0.219046,0.199788,0.965128,0.355303,-0.080834


In [58]:
tar_xpr_meta.index = tar_xpr_meta['sig_id'].values.copy()
xpr_adata.obs = tar_xpr_meta.loc[xpr_adata.obs_names,:].copy()

In [59]:
xpr_adata

AnnData object with n_obs × n_vars = 140945 × 12328
    obs: 'bead_batch', 'nearest_dose', 'pert_dose', 'pert_dose_unit', 'pert_idose', 'pert_itime', 'pert_time', 'pert_time_unit', 'cell_mfc_name', 'pert_mfc_id', 'nsample', 'cc_q75', 'ss_ngene', 'tas', 'pct_self_rank_q25', 'wt', 'median_recall_rank_spearman', 'median_recall_rank_wtcs_50', 'median_recall_score_spearman', 'median_recall_score_wtcs_50', 'batch_effect_tstat', 'batch_effect_tstat_pct', 'is_hiq', 'qc_pass', 'pert_id', 'sig_id', 'pert_type', 'cell_iname', 'det_wells', 'det_plates', 'distil_ids', 'build_name', 'project_code', 'cmap_name', 'is_exemplar_sig', 'is_ncs_sig', 'is_null_sig'
    var: 'gene_id', 'gene_symbol', 'ensembl_id', 'gene_title', 'gene_type', 'src', 'feature_space'

In [60]:
xpr_adata.write('/public/home/caojun/project/RUSH/3_work/input/cmap_xpr_ad.h5ad')

### 2.3.2 unique

In [ ]:
# 每个扰动只保留一个代表！！！

In [61]:
tar_xpr_meta_sub = tar_xpr_meta.loc[:,['sig_id','project_code','cell_iname','pert_id','cmap_name', 'pert_dose', 'pert_time']]

In [65]:
tar_xpr_meta_sub['pert_time'].value_counts()

pert_time
 96.0     140847
 192.0        69
 408.0        24
-666.0         5
Name: count, dtype: int64

In [64]:
tar_xpr_meta_sub['pert_dose'].value_counts()

pert_dose
20.0    69
2.0      5
Name: count, dtype: int64

In [67]:
xpr666 = tar_xpr_meta_sub.index[tar_xpr_meta_sub['pert_time']==-666.0].to_list()
len(xpr666)

5

In [68]:
tar_xpr_meta_sub = tar_xpr_meta_sub.loc[~tar_xpr_meta_sub.index.isin(xpr666),:]

In [69]:
tar_xpr_meta_sub

,sig_id,project_code,cell_iname,pert_id,cmap_name,pert_dose,pert_time
HAHN001_ES2_96H:G06,HAHN001_ES2_96H:G06,HAHN,ES2,HAHN-000061,AURKB,NaN,96.0
HAHN001_HCC44_96H:E15,HAHN001_HCC44_96H:E15,HAHN,HCC44,BRDN0001062183,RAC1,NaN,96.0
HAHN001_HCC44_96H:K22,HAHN001_HCC44_96H:K22,HAHN,HCC44,BRDN0000734411,KIF11,NaN,96.0
HAHN001_HCC44_96H:L09,HAHN001_HCC44_96H:L09,HAHN,HCC44,HAHN-000098,XBP,NaN,96.0
HAHN001_HCC44_96H:H08,HAHN001_HCC44_96H:H08,HAHN,HCC44,HAHN-000086,ORC4,NaN,96.0
...,...,...,...,...,...,...,...
XPR044_LCLC103H.311_96H:L21,XPR044_LCLC103H.311_96H:L21,XPR,LCLC103H,BRDN0001489479,CCNA2,NaN,96.0
XPR044_LCLC103H.311_96H:H03,XPR044_LCLC103H.311_96H:H03,XPR,LCLC103H,BRDN0001486533,NaN,NaN,96.0
XPR044_LCLC103H.311_96H:O24,XPR044_LCLC103H.311_96H:O24,XPR,LCLC103H,BRDN0001482620,NaN,NaN,96.0
XPR044_LCLC103H.311_96H:E06,XPR044_LCLC103H.311_96H:E06,XPR,LCLC103H,BRDN0001146116,CKS2,NaN,96.0


In [70]:
pd.crosstab(
    xpr_adata.obs['cell_iname'],
    xpr_adata.obs['pert_id']
)

pert_id,BRDN0000562842,BRDN0000562885,BRDN0000562897,BRDN0000562936,BRDN0000562942,BRDN0000563033,BRDN0000563442,BRDN0000563480,BRDN0000563482,BRDN0000563485,...,HAHN-000272,HDAC1-SHRNA2,HDAC1-SHRNA4,HDAC1-SHRNA5,NIPBL,PDS5B,RAD21,SMC1A,SMC3,STAG2
cell_iname,,,,,,,,,,,,,,,,,,,,,
A375,0,1,1,1,1,1,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0
A549,1,1,2,2,1,2,1,1,1,1,...,1,0,0,0,0,0,0,0,0,0
AGS,0,1,1,1,1,1,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0
BICR6,0,1,1,1,1,1,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0
DANG,1,0,1,1,0,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
DLD1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
ES2,1,1,2,2,1,2,1,1,1,1,...,1,0,0,0,0,0,0,0,0,0
HCC44,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
HCC1806,1,0,1,1,0,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [71]:
xpr_adata = xpr_adata[~xpr_adata.obs_names.isin(xpr666),:]

In [84]:
xpr_adata.obs['pert_time'].value_counts()

pert_time
96.0     140847
192.0        69
408.0        24
Name: count, dtype: int64

In [83]:
# 使用正则表达式提取数值部分（去掉可能的 ' h' 或其他单位）
xpr_adata.obs['pert_itime'] = xpr_adata.obs['pert_itime'].str.extract(r'([\d.]+)').astype(float)

# 填充缺失值为 0
xpr_adata.obs['pert_itime'] = xpr_adata.obs['pert_itime'].fillna(0)

# 确保转换为浮动类型
xpr_adata.obs['pert_itime'] = xpr_adata.obs['pert_itime'].astype(float)


/tmp/ipykernel_30967/2709207368.py:2: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  xpr_adata.obs['pert_itime'] = xpr_adata.obs['pert_itime'].str.extract(r'([\d.]+)').astype(float)


In [85]:
xpr_adata.obs['pert_dose'] = xpr_adata.obs['pert_dose'].fillna(0).astype(float)

In [86]:
import numpy as np

# 将 pert_dose 和 pert_itime 转换为数值型数据，提取最大剂量和最大时间
#cp_adata.obs['pert_dose'] = cp_adata.obs['pert_dose'].str.extract(r'([\d.]+)', expand=False).fillna(0).astype(float)

# 通过 groupby 来按细胞系和药物分组
def get_max_values(group):
    max_dose = group['pert_dose'].max()
    max_itime = group['pert_itime'].max()
    
    # 找到最大剂量和最大时间的索引，确保有符合条件的行
    max_row = group[(group['pert_dose'] == max_dose) & (group['pert_itime'] == max_itime)]
    
    if not max_row.empty:  # 确保 max_row 非空
        return max_row.index[0]  # 返回第一行的索引（即代表值）
    else:
        return np.nan  # 如果没有符合条件的行，返回 NaN

# 获取每个分组的代表值
representative_ids = xpr_adata.obs.groupby(['pert_id', 'cell_iname']).apply(get_max_values)

# 删除 NaN 行，因为它们没有代表值
representative_ids = representative_ids.dropna()

# 将代表值的 hit 标记为 True
xpr_adata.obs['hit'] = xpr_adata.obs.index.isin(representative_ids)


/tmp/ipykernel_30967/1499037269.py:20: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  representative_ids = xpr_adata.obs.groupby(['pert_id', 'cell_iname']).apply(get_max_values)
/tmp/ipykernel_30967/1499037269.py:20: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  representative_ids = xpr_adata.obs.groupby(['pert_id', 'cell_iname']).apply(get_max_values)


In [87]:
xpr_adata.obs['hit'].value_counts()

hit
True     132464
False      8476
Name: count, dtype: int64

In [89]:
xpr_adata[xpr_adata.obs['hit'],:]

View of AnnData object with n_obs × n_vars = 132464 × 12328
    obs: 'bead_batch', 'nearest_dose', 'pert_dose', 'pert_dose_unit', 'pert_idose', 'pert_itime', 'pert_time', 'pert_time_unit', 'cell_mfc_name', 'pert_mfc_id', 'nsample', 'cc_q75', 'ss_ngene', 'tas', 'pct_self_rank_q25', 'wt', 'median_recall_rank_spearman', 'median_recall_rank_wtcs_50', 'median_recall_score_spearman', 'median_recall_score_wtcs_50', 'batch_effect_tstat', 'batch_effect_tstat_pct', 'is_hiq', 'qc_pass', 'pert_id', 'sig_id', 'pert_type', 'cell_iname', 'det_wells', 'det_plates', 'distil_ids', 'build_name', 'project_code', 'cmap_name', 'is_exemplar_sig', 'is_ncs_sig', 'is_null_sig', 'hit'
    var: 'gene_id', 'gene_symbol', 'ensembl_id', 'gene_title', 'gene_type', 'src', 'feature_space'

In [90]:
xpr_adata[xpr_adata.obs['hit'],:].write('/public/home/caojun/project/RUSH/3_work/input/cmap_xpr_ad_unique.h5ad')

### 2.3.3 split

In [91]:
xpr_adata = sc.read_h5ad('/public/home/caojun/project/RUSH/3_work/input/cmap_xpr_ad_unique.h5ad')

In [92]:
xpr_adata

AnnData object with n_obs × n_vars = 132464 × 12328
    obs: 'bead_batch', 'nearest_dose', 'pert_dose', 'pert_dose_unit', 'pert_idose', 'pert_itime', 'pert_time', 'pert_time_unit', 'cell_mfc_name', 'pert_mfc_id', 'nsample', 'cc_q75', 'ss_ngene', 'tas', 'pct_self_rank_q25', 'wt', 'median_recall_rank_spearman', 'median_recall_rank_wtcs_50', 'median_recall_score_spearman', 'median_recall_score_wtcs_50', 'batch_effect_tstat', 'batch_effect_tstat_pct', 'is_hiq', 'qc_pass', 'pert_id', 'sig_id', 'pert_type', 'cell_iname', 'det_wells', 'det_plates', 'distil_ids', 'build_name', 'project_code', 'cmap_name', 'is_exemplar_sig', 'is_ncs_sig', 'is_null_sig', 'hit'
    var: 'gene_id', 'gene_symbol', 'ensembl_id', 'gene_title', 'gene_type', 'src', 'feature_space'

In [93]:
output_dir = "/public/home/caojun/project/RUSH/3_work/output/store/xpr_split"

In [94]:
# 确定每个文件保存多少行数据
chunk_size = 1000

# 计算切分的数量
n_chunks = np.ceil(len(xpr_adata.obs) / chunk_size).astype(int)

# 切分并保存
for i in range(n_chunks):
    # 切分数据，取第 i 个区间的 1000 行
    start_idx = i * chunk_size
    end_idx = min((i + 1) * chunk_size, len(xpr_adata.obs))
    
    # 创建一个新的 AnnData 子集
    xpr_adata_sub = xpr_adata[start_idx:end_idx].copy()
    
    # 保存为 .h5ad 文件
    file_name = f"xpr_split_{i}.h5ad"
    file_path = os.path.join(output_dir, file_name)
    
    # 写入文件
    xpr_adata_sub.write(file_path)

    print(f"Saved {file_name} to {file_path}")


Saved xpr_split_0.h5ad to /public/home/caojun/project/RUSH/3_work/output/store/xpr_split/xpr_split_0.h5ad
Saved xpr_split_1.h5ad to /public/home/caojun/project/RUSH/3_work/output/store/xpr_split/xpr_split_1.h5ad
Saved xpr_split_2.h5ad to /public/home/caojun/project/RUSH/3_work/output/store/xpr_split/xpr_split_2.h5ad
Saved xpr_split_3.h5ad to /public/home/caojun/project/RUSH/3_work/output/store/xpr_split/xpr_split_3.h5ad
Saved xpr_split_4.h5ad to /public/home/caojun/project/RUSH/3_work/output/store/xpr_split/xpr_split_4.h5ad
Saved xpr_split_5.h5ad to /public/home/caojun/project/RUSH/3_work/output/store/xpr_split/xpr_split_5.h5ad
Saved xpr_split_6.h5ad to /public/home/caojun/project/RUSH/3_work/output/store/xpr_split/xpr_split_6.h5ad
Saved xpr_split_7.h5ad to /public/home/caojun/project/RUSH/3_work/output/store/xpr_split/xpr_split_7.h5ad
Saved xpr_split_8.h5ad to /public/home/caojun/project/RUSH/3_work/output/store/xpr_split/xpr_split_8.h5ad
Saved xpr_split_9.h5ad to /public/home/caojun/